# Deep Learning Lab: Variational Autoencoders (VAEs)
##### Prof. Dr. Heinke Hihn · IU International University of Applied Sciences · CSE Deep Learning Course

## Learning Objectives

By the end of this lab, you will be able to:

1. Explain why a VAE learns **distributions** instead of point embeddings
2. Implement a convolutional VAE and the reparameterization trick in PyTorch
3. Derive and interpret the **Evidence Lower Bound (ELBO)**
4. Separate reconstruction loss from KL regularization during training
5. Visualize, sample, and interpolate in a learned latent space
6. Apply VAEs to generation, representation learning, and anomaly detection

## Prerequisites

- Feedforward and convolutional neural networks
- Backpropagation and probability basics
- Familiarity with PyTorch

This lab is deliberately practical. We use MNIST so that experiments run on a laptop and the two-dimensional latent space can be visualized directly.

## Part 1: Setup and Data

We will train on a subset by default. Set `FAST_MODE = False` for the complete dataset and increase the number of epochs later in the notebook.

In [ ]:
# If necessary, install the dependencies in your environment:
# %pip install torch torchvision matplotlib tqdm certifi

import random
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

plt.style.use("seaborn-v0_8-whitegrid")
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

In [ ]:
FAST_MODE = True
BATCH_SIZE = 128

# Python installations on macOS sometimes do not know where to find the
# system CA certificates. Certifi supplies a current trusted CA bundle.
# We keep HTTPS verification enabled; do not use an unverified SSL context.
import ssl
try:
    import certifi
except ImportError as error:
    raise ImportError(
        "Install the certificate bundle with `%pip install certifi`, then restart the kernel."
    ) from error

ssl._create_default_https_context = (
    lambda: ssl.create_default_context(cafile=certifi.where())
)

transform = transforms.ToTensor()  # pixels are in [0, 1]
try:
    train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
    test_data = datasets.MNIST("./data", train=False, download=True, transform=transform)
except RuntimeError as error:
    if "CERTIFICATE_VERIFY_FAILED" in str(error):
        raise RuntimeError(
            "SSL certificates are still unavailable. On a python.org macOS installation, "
            "run 'Install Certificates.command' from the Python 3.13 folder in Applications, "
            "restart Jupyter, and run this cell again."
        ) from error
    raise

if FAST_MODE:
    generator = torch.Generator().manual_seed(SEED)
    train_ids = torch.randperm(len(train_data), generator=generator)[:20_000]
    test_ids = torch.randperm(len(test_data), generator=generator)[:5_000]
    train_data = Subset(train_data, train_ids)
    test_data = Subset(test_data, test_ids)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print(f"Training samples: {len(train_data):,}")
print(f"Test samples:     {len(test_data):,}")
print(f"Batch shape:      {tuple(images.shape)}")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, image, label in zip(axes.flat, images[:12], labels[:12]):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"label = {label.item()}")
    ax.axis("off")
plt.suptitle("A mini-batch of MNIST digits")
plt.tight_layout()
plt.show()

## Part 2: From Autoencoder to Variational Autoencoder

A standard autoencoder maps an input to one point $z$ and reconstructs the input:

$$x \xrightarrow{\text{encoder}} z \xrightarrow{\text{decoder}} \hat{x}$$

It can reconstruct well, but empty or irregular regions in latent space make random sampling unreliable.

A VAE encoder instead predicts the parameters of a distribution:

$$q_\phi(z\mid x)=\mathcal{N}\left(\mu_\phi(x),\operatorname{diag}(\sigma_\phi^2(x))\right).$$

We sample $z$ from that distribution and ask the decoder to model $p_\theta(x\mid z)$. A prior, usually $p(z)=\mathcal{N}(0,I)$, encourages a smooth, sampleable latent space.

### Practical uses

- **Generation:** sample new images, molecules, designs, or signals
- **Representation learning:** obtain compact features for clustering or downstream models
- **Anomaly detection:** flag inputs that reconstruct poorly or fit the latent model badly
- **Compression and denoising:** encode information compactly and reconstruct plausible inputs
- **Data augmentation:** generate additional examples, with careful validation of fidelity and bias

## Part 3: Understanding the ELBO

Directly maximizing the log evidence $\log p_\theta(x)$ is difficult because it requires integrating over every possible $z$. Introducing the approximate posterior $q_\phi(z\mid x)$ gives

$$
\log p_\theta(x)
= \underbrace{\mathbb{E}_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)]
- D_{KL}\!\left(q_\phi(z\mid x)\,\|\,p(z)\right)}_{\text{ELBO}(x)}
+ D_{KL}\!\left(q_\phi(z\mid x)\,\|\,p_\theta(z\mid x)\right).
$$

The final KL divergence is non-negative, so the ELBO is a **lower bound** on $\log p_\theta(x)$.

We maximize the ELBO—or equivalently minimize the negative ELBO:

$$
\mathcal{L}_{VAE}
= \underbrace{-\mathbb{E}_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)]}_{\text{reconstruction loss}}
+ \beta\underbrace{D_{KL}(q_\phi(z\mid x)\|p(z))}_{\text{latent regularization}}.
$$

For binary-valued pixels, the first term is binary cross-entropy. For diagonal Gaussians and a standard-normal prior,

$$D_{KL}=-\frac{1}{2}\sum_j\left(1+\log\sigma_j^2-\mu_j^2-\sigma_j^2\right).$$

`beta=1` is the standard VAE objective. Larger values usually impose a more organized latent space but can make reconstructions worse. This is a real engineering trade-off, not a free improvement.

In [ ]:
# Visualize how the KL term penalizes distance from N(0, 1)
mu_values = torch.linspace(-3, 3, 200)
logvar_choices = [-2.0, 0.0, 1.0]  # variance ≈ 0.14, 1.00, 2.72

plt.figure(figsize=(8, 4))
for logvar in logvar_choices:
    kl = -0.5 * (1 + logvar - mu_values.pow(2) - np.exp(logvar))
    plt.plot(mu_values, kl, label=f"log variance = {logvar:+.1f}")
plt.xlabel("posterior mean μ")
plt.ylabel("KL(q(z|x) || N(0,1))")
plt.title("The KL term is smallest at μ = 0 and variance = 1")
plt.legend()
plt.show()

### Exercise 1: Read the KL Plot

1. Why is the penalty smallest when $\mu=0$ and $\sigma^2=1$?
2. What would happen if we trained using only reconstruction loss?
3. What would happen if the KL term dominated from the first update?

<details>
<summary>Click here for discussion</summary>

1. Then the approximate posterior exactly matches the standard-normal prior.
2. The model becomes similar to a noisy autoencoder: reconstruction may be good, but random prior samples can land in unsupported regions.
3. The encoder may ignore the input and output the prior for every example. This is called <em>posterior collapse</em>. KL warm-up is one possible mitigation.

</details>

## Part 4: The Reparameterization Trick

Naively sampling $z\sim\mathcal{N}(\mu,\sigma^2)$ interrupts gradient-based learning. We move the randomness into an auxiliary variable:

$$\epsilon\sim\mathcal{N}(0,I),\qquad z=\mu+\sigma\odot\epsilon,$$

with $\sigma=\exp(\tfrac{1}{2}\log\sigma^2)$. Now $z$ is a differentiable function of the encoder outputs and random noise.

In [ ]:
def reparameterize_demo(mu, logvar, n_samples=2_000):
    # Draw differentiable samples from a diagonal Gaussian.
    std = torch.exp(0.5 * logvar)
    eps = torch.randn(n_samples, mu.numel())
    return mu + eps * std

mu_demo = torch.tensor([-1.0, 1.5])
logvar_demo = torch.log(torch.tensor([0.25, 1.0]))
z_demo = reparameterize_demo(mu_demo, logvar_demo)

print("Target mean:    ", mu_demo.numpy())
print("Empirical mean: ", z_demo.mean(0).numpy().round(2))
print("Target std:     ", torch.exp(0.5 * logvar_demo).numpy())
print("Empirical std:  ", z_demo.std(0).numpy().round(2))

plt.figure(figsize=(5, 5))
plt.scatter(z_demo[:, 0], z_demo[:, 1], s=8, alpha=0.25)
plt.xlabel("z₁")
plt.ylabel("z₂")
plt.title("Samples produced by reparameterization")
plt.axis("equal")
plt.show()

## Part 5: Build a Convolutional VAE

The encoder reduces a $28\times28$ image to two vectors, `mu` and `logvar`. The decoder maps a sampled latent vector back to image logits.

We use **logits**, not sigmoid probabilities, in the model because `binary_cross_entropy_with_logits` is numerically more stable.

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim

        # 1 x 28 x 28 -> 32 x 14 x 14 -> 64 x 7 x 7
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, 64 * 7 * 7)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # EXERCISE: explain why 0.5 appears here.
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(self.decoder_input(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


model = ConvVAE(latent_dim=2).to(device)
with torch.no_grad():
    logits, mu, logvar = model(images[:8].to(device))

print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input: {tuple(images[:8].shape)} -> logits: {tuple(logits.shape)}")
print(f"mu: {tuple(mu.shape)}, logvar: {tuple(logvar.shape)}")

### Exercise 2: Implement the Negative ELBO

Complete the function below on your own. Use a **sum over pixels and latent dimensions**, then divide all terms by batch size so the two contributions stay on a comparable scale.

In [ ]:
def student_vae_loss(logits, x, mu, logvar, beta=1.0):
    # TODO: implement your solution here
    pass

<details>
<summary>💡 Click here for the solution</summary>

```python
def student_vae_loss(logits, x, mu, logvar, beta=1.0):
    reconstruction = F.binary_cross_entropy_with_logits(logits, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    batch_size = x.size(0)
    reconstruction = reconstruction / batch_size
    kl = kl / batch_size
    return reconstruction + beta * kl, reconstruction, kl
```

The training code below uses the completed version so the notebook remains runnable.

</details>

In [ ]:
def vae_loss(logits, x, mu, logvar, beta=1.0):
    # Negative ELBO, averaged over the batch.
    reconstruction = F.binary_cross_entropy_with_logits(logits, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    batch_size = x.size(0)
    reconstruction = reconstruction / batch_size
    kl = kl / batch_size
    total = reconstruction + beta * kl
    return total, reconstruction, kl


# Unit check: matching the prior should produce KL = 0
zeros = torch.zeros(4, 2)
dummy_logits = torch.zeros(4, 1, 28, 28)
dummy_x = torch.zeros_like(dummy_logits)
_, _, kl_check = vae_loss(dummy_logits, dummy_x, zeros, zeros)
assert torch.allclose(kl_check, torch.tensor(0.0)), "KL formula is incorrect"
print("ELBO implementation check passed.")

## Part 6: Train and Monitor Both ELBO Terms

Looking only at total loss hides the central VAE trade-off. We therefore track reconstruction and KL terms separately.

We also apply a short **KL warm-up**: `beta` grows from 0 to 1. Early training focuses on learning useful reconstructions before applying full latent regularization.

In [ ]:
def run_epoch(model, loader, beta, optimizer=None):
    training = optimizer is not None
    model.train(training)
    totals = np.zeros(3, dtype=float)
    examples_seen = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for x, _ in tqdm(loader, leave=False, desc="train" if training else "valid"):
            x = x.to(device)
            if training:
                optimizer.zero_grad()

            logits, mu, logvar = model(x)
            loss, recon, kl = vae_loss(logits, x, mu, logvar, beta)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = x.size(0)
            totals += np.array([loss.item(), recon.item(), kl.item()]) * batch_size
            examples_seen += batch_size

    return totals / examples_seen


EPOCHS = 5 if FAST_MODE else 10
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = {key: [] for key in ["train_total", "train_recon", "train_kl",
                                "valid_total", "valid_recon", "valid_kl", "beta"]}

start = time.time()
for epoch in range(1, EPOCHS + 1):
    beta = min(1.0, epoch / 3)  # KL warm-up
    train_metrics = run_epoch(model, train_loader, beta, optimizer)
    valid_metrics = run_epoch(model, test_loader, beta)

    history["beta"].append(beta)
    for split, values in [("train", train_metrics), ("valid", valid_metrics)]:
        for name, value in zip(["total", "recon", "kl"], values):
            history[f"{split}_{name}"].append(value)

    print(f"Epoch {epoch:02d}/{EPOCHS} | beta={beta:.2f} | "
          f"train: total={train_metrics[0]:.2f}, recon={train_metrics[1]:.2f}, KL={train_metrics[2]:.2f} | "
          f"valid: total={valid_metrics[0]:.2f}")

print(f"Training time: {time.time() - start:.1f} seconds")

In [ ]:
epochs = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history["train_total"], "o-", label="train")
axes[0].plot(epochs, history["valid_total"], "o-", label="validation")
axes[0].set_title("Negative ELBO")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs, history["valid_recon"], "o-", label="reconstruction")
axes[1].plot(epochs, history["valid_kl"], "o-", label="KL")
axes[1].set_title("Validation loss decomposition")
axes[1].set_xlabel("epoch")
axes[1].legend()

axes[2].plot(epochs, history["beta"], "o-", color="tab:purple")
axes[2].set_ylim(0, 1.1)
axes[2].set_title("KL warm-up schedule")
axes[2].set_xlabel("epoch")
axes[2].set_ylabel("beta")

plt.tight_layout()
plt.show()

### Exercise 3: Diagnose the Training Curves

- Why is reconstruction loss numerically much larger than KL here?
- Did validation negative ELBO improve?
- Is the KL term close to zero? If so, investigate posterior collapse.
- Retrain with `beta = 0`, `beta = 0.25`, or `beta = 4`. Compare reconstruction quality and latent organization.

**Important:** ELBO values from different implementations are comparable only if reduction and likelihood conventions match. Always report whether terms are summed or averaged over pixels and batches.

## Part 7: Reconstruction and Generation

Reconstruction asks: “Can the model preserve information about a given input?” Generation asks: “Does sampling from the prior produce plausible new inputs?” A model can perform well on one and poorly on the other.

In [ ]:
model.eval()
x, _ = next(iter(test_loader))
x = x[:10].to(device)
with torch.no_grad():
    logits, _, _ = model(x)
    reconstructions = torch.sigmoid(logits)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for i in range(10):
    axes[0, i].imshow(x[i].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[1, i].imshow(reconstructions[i].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[0, i].axis("off")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("input", fontsize=11)
axes[1, 0].set_ylabel("reconstruction", fontsize=11)
plt.suptitle("Inputs and stochastic VAE reconstructions")
plt.tight_layout()
plt.show()

In [ ]:
# Sample entirely new digits from p(z) = N(0, I)
with torch.no_grad():
    z = torch.randn(25, model.latent_dim, device=device)
    samples = torch.sigmoid(model.decode(z)).cpu()

fig, axes = plt.subplots(5, 5, figsize=(7, 7))
for ax, sample in zip(axes.flat, samples):
    ax.imshow(sample.squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
plt.suptitle("Novel samples drawn from the prior")
plt.tight_layout()
plt.show()

## Part 8: Inspect the Latent Space

Labels were never shown to the VAE. If digits with similar shapes occupy nearby regions, that structure emerged from reconstruction plus regularization—not from a classification objective.

In [ ]:
@torch.no_grad()
def collect_latents(model, loader):
    model.eval()
    mus, labels = [], []
    for x, y in loader:
        mu, _ = model.encode(x.to(device))
        mus.append(mu.cpu())
        labels.append(y)
    return torch.cat(mus), torch.cat(labels)


latent_mu, latent_labels = collect_latents(model, test_loader)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(latent_mu[:, 0], latent_mu[:, 1], c=latent_labels,
                      cmap="tab10", s=8, alpha=0.6)
plt.colorbar(scatter, ticks=range(10), label="digit")
plt.xlabel("latent mean z₁")
plt.ylabel("latent mean z₂")
plt.title("Two-dimensional latent representation")
plt.show()

In [ ]:
# Decode a regular grid to see how output changes across latent space
grid_size = 15
coordinates = torch.linspace(-3, 3, grid_size)
canvas = torch.zeros(grid_size * 28, grid_size * 28)

model.eval()
with torch.no_grad():
    for row, z2 in enumerate(reversed(coordinates)):
        for col, z1 in enumerate(coordinates):
            z = torch.tensor([[z1, z2]], device=device)
            digit = torch.sigmoid(model.decode(z)).cpu().squeeze()
            canvas[row*28:(row+1)*28, col*28:(col+1)*28] = digit

plt.figure(figsize=(10, 10))
plt.imshow(canvas, cmap="gray", extent=[-3, 3, -3, 3])
plt.xlabel("z₁")
plt.ylabel("z₂")
plt.title("Decoder output across latent space")
plt.show()

## Part 9: Latent Interpolation

Interpolation is useful for controlled generation and for testing whether the learned space changes smoothly. We interpolate between the **posterior means** of two real digits.

In [ ]:
all_test_images, all_test_labels = next(iter(DataLoader(test_data, batch_size=len(test_data))))
start_idx = int((all_test_labels == 1).nonzero()[0])
end_idx = int((all_test_labels == 8).nonzero()[0])
endpoints = all_test_images[[start_idx, end_idx]].to(device)

with torch.no_grad():
    endpoint_mu, _ = model.encode(endpoints)
    alphas = torch.linspace(0, 1, 12, device=device).unsqueeze(1)
    z_path = (1 - alphas) * endpoint_mu[0] + alphas * endpoint_mu[1]
    path_images = torch.sigmoid(model.decode(z_path)).cpu()

fig, axes = plt.subplots(1, 12, figsize=(16, 2))
for alpha, ax, image in zip(alphas.cpu(), axes, path_images):
    ax.imshow(image.squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{alpha.item():.1f}")
    ax.axis("off")
plt.suptitle("Linear interpolation in latent space: digit 1 → digit 8")
plt.tight_layout()
plt.show()

## Part 10: Practical Use Case—Anomaly Scoring

Suppose a quality-control system is trained mostly on normal patterns. A simple VAE anomaly score is the per-sample negative ELBO. Here, rotated digits act as synthetic anomalies.

This is a teaching demonstration, not a production detector: reconstruction models may also reconstruct anomalies well. Thresholds must be chosen using labeled validation data, and performance should be reported with metrics such as precision-recall curves.

In [ ]:
@torch.no_grad()
def negative_elbo_per_sample(model, x):
    model.eval()
    logits, mu, logvar = model(x)
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="none")
    recon = recon.flatten(1).sum(1)
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(1)
    return recon + kl


normal, _ = next(iter(test_loader))
normal = normal[:128].to(device)
rotated = torch.stack([TF.rotate(image.cpu(), angle=45) for image in normal]).to(device)

normal_scores = negative_elbo_per_sample(model, normal).cpu().numpy()
rotated_scores = negative_elbo_per_sample(model, rotated).cpu().numpy()

plt.figure(figsize=(8, 4))
plt.hist(normal_scores, bins=25, alpha=0.65, label="normal digits")
plt.hist(rotated_scores, bins=25, alpha=0.65, label="rotated digits")
plt.xlabel("negative ELBO (larger = more anomalous)")
plt.ylabel("count")
plt.title("VAE anomaly-score distributions")
plt.legend()
plt.show()

print(f"Mean normal score:  {normal_scores.mean():.1f}")
print(f"Mean rotated score: {rotated_scores.mean():.1f}")

### Exercise 4: Build a Simple Detector

1. Use the 95th percentile of `normal_scores` as a threshold.
2. Compute the fraction of normal and rotated images flagged as anomalies.
3. Inspect false positives and false negatives visually.
4. Explain why selecting the threshold on the final test set would be data leakage.

<details>
<summary>Click here for a starting solution</summary>

```python
threshold = np.percentile(normal_scores, 95)
normal_flag_rate = (normal_scores > threshold).mean()
rotated_detection_rate = (rotated_scores > threshold).mean()
print(f"Threshold: {threshold:.1f}")
print(f"Normal flag rate: {normal_flag_rate:.1%}")
print(f"Rotated detection rate: {rotated_detection_rate:.1%}")
```

In a real project, estimate the threshold on a separate validation set. Reserve the test set for one final, unbiased evaluation.

</details>

## Part 11: From Lab to Real Projects

| Use case | What the VAE provides | What to validate |
|---|---|---|
| Synthetic data | samples from a learned distribution | realism, diversity, privacy, bias |
| Anomaly detection | reconstruction/KL-based score | detection metrics, threshold stability, drift |
| Compression | low-dimensional stochastic code | rate–distortion trade-off |
| Representation learning | compact latent features | downstream performance and robustness |
| Denoising/imputation | distribution of plausible reconstructions | calibration and domain constraints |

### Design choices to explore

- **Latent dimension:** 2 is visualizable; 16–128 is often more useful.
- **Likelihood:** Bernoulli is convenient for MNIST; continuous or natural images need a more appropriate observation model.
- **Beta:** controls reconstruction versus regularization.
- **Architecture:** convolutional, recurrent, graph, or transformer encoders/decoders match different data types.
- **Conditional VAE:** condition encoder and decoder on a class or property for controllable generation.

### Final challenge

Choose one extension and report both quantitative and qualitative results:

1. Train with three beta values and compare reconstruction, KL, and generated samples.
2. Increase `latent_dim` and train a classifier on frozen latent means.
3. Build a conditional VAE that generates a requested digit.
4. Create a validation/test protocol for anomaly detection with several anomaly types.

## Key Takeaways

1. A VAE learns an approximate posterior $q_\phi(z\mid x)$ and a decoder $p_\theta(x\mid z)$.
2. The ELBO balances data reconstruction against closeness to a latent prior.
3. The reparameterization trick makes stochastic sampling compatible with backpropagation.
4. Reconstruction, random generation, and latent organization must be evaluated separately.
5. VAEs are useful practical models, but the likelihood, beta, architecture, and evaluation protocol must match the use case.

### Suggested reading

- Kingma & Welling (2014), *Auto-Encoding Variational Bayes*
- Rezende, Mohamed & Wierstra (2014), *Stochastic Backpropagation and Approximate Inference in Deep Generative Models*
- Higgins et al. (2017), *beta-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework*